# 20 — Report readiness check

Fail loudly if the report bundle is missing core evidence, if only provisional seed trade exists, or if the price hypothesis has not been analysed.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

import json


In [ ]:
manifest_path = PATHS.report_inputs / "report_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError("Run notebook 19 first.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
checks = []
checks.append({"check": "core_report_bundle_complete", "passed": bool(manifest.get("complete")), "detail": str(manifest.get("missing", []))})
trade = pd.read_csv(PATHS.processed / "fuel_trade_annual.csv")
seed_only = "status" in trade.columns and set(trade["status"].dropna()) == {"seed_provisional"}
checks.append({"check": "trade_not_seed_only", "passed": not seed_only, "detail": "Re-run JODI/DGEG acquisition before publication" if seed_only else "downloaded/cross-checked trade available"})
checks.append({"check": "price_models_available", "passed": (PATHS.metrics / "price_pass_through_models.csv").exists(), "detail": "Price-exposure claim must remain untested if false"})
checks.append({"check": "dgeg_trade_reconciled", "passed": (PATHS.metrics / "jodi_dgeg_trade_reconciliation.csv").exists(), "detail": "Strongly recommended before final report"})
readiness = pd.DataFrame(checks)
persist_dataframe(readiness, PATHS.metrics / "report_readiness.csv", key_columns=["check"])
display(readiness)
if not readiness.loc[readiness["check"].isin(["core_report_bundle_complete", "trade_not_seed_only"]), "passed"].all():
    raise RuntimeError("Core publication-readiness checks failed. See data/metrics/report_readiness.csv")


A failed price-model check does not block a purely physical-supply report, but it **does** block any conclusion that refining reconfiguration changed domestic price exposure.
